# CASMI 2026: offline spectral-retrieval baseline

Attach the competition data, a source snapshot containing `src/` and `requirements.txt`,
and a prebuilt `library.sqlite`. Edit the paths below to match the actual Kaggle mounts.
Optionally attach compatible wheels for the exact locked dependencies. This notebook
uses no network access. It produces `submission.csv` and a diagnostic report.

This baseline retrieves training structures; it does not generate novel molecules.
The visible test file overlaps training, so local lookup success is not a hidden-test score.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

# Adjust these defaults to your attached Kaggle input paths.
SOURCE_ROOT = Path(os.environ.get("CASMI_SOURCE_ROOT", "/kaggle/input/casmi-source"))
DATA_ROOT = Path(os.environ.get(
    "CASMI_DATA_ROOT", "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"
))
INDEX_PATH = Path(os.environ.get("CASMI_INDEX_PATH", "/kaggle/input/casmi-reference/library.sqlite"))
OUTPUT = Path(os.environ.get("CASMI_OUTPUT", "/kaggle/working/submission.csv"))
WHEELHOUSE = os.environ.get("CASMI_WHEELHOUSE")  # Or set to your attached wheels directory.
DEPS_DIR = OUTPUT.parent / "casmi-deps"

for required in [SOURCE_ROOT / "src/casmi/__init__.py", DATA_ROOT / "test.parquet", INDEX_PATH]:
    if not required.is_file():
        raise FileNotFoundError(f"Required input missing; configure the paths above: {required}")


In [ ]:
# Run package code in fresh child processes to avoid notebook-kernel import state.
runtime_env = os.environ.copy()
python_paths = [str(SOURCE_ROOT / "src")]
if WHEELHOUSE:
    wheels = Path(WHEELHOUSE)
    requirements = SOURCE_ROOT / "requirements.txt"
    if not wheels.is_dir() or not requirements.is_file():
        raise FileNotFoundError("Attach a wheel directory and the source's requirements.txt")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--no-index", "--only-binary=:all:",
        "--require-hashes", "--ignore-installed", "--target", str(DEPS_DIR),
        "--find-links", str(wheels), "-r", str(requirements),
    ], check=True)
    python_paths.insert(0, str(DEPS_DIR))
if runtime_env.get("PYTHONPATH"):
    python_paths.append(runtime_env["PYTHONPATH"])
runtime_env["PYTHONPATH"] = os.pathsep.join(python_paths)
subprocess.run([
    sys.executable, "-c",
    "import numpy, pyarrow; from casmi.chemistry import require_metric_rdkit; "
    "require_metric_rdkit(); print('Runtime dependencies ready')",
], check=True, env=runtime_env)


In [ ]:
subprocess.run([
    sys.executable, "-m", "casmi", "predict",
    "--library", str(INDEX_PATH), "--test", str(DATA_ROOT / "test.parquet"),
    "--output", str(OUTPUT),
], check=True, env=runtime_env)

subprocess.run([
    sys.executable, "-m", "casmi", "check-submission",
    "--submission", str(OUTPUT), "--test", str(DATA_ROOT / "test.parquet"),
], check=True, env=runtime_env)
print(f"Submission written: {OUTPUT}")
print(f"Diagnostics: {OUTPUT.with_suffix('.report.json')}")
